# DSpace DB5 load

Notebook de prueba para el nodo `load_dspacedb5_tables` del pipeline `load_dspacedb5`.

Lee snapshots desde `raw/dspacedb5/{table}#parquet`, normaliza metadata de extraccion, agrega `_load_datetime` y deja los DataFrames listos para cargar en `ldg/dspacedb5/{table}`. Esta notebook no hace `catalog.save`.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)


In [ ]:
tables = [
    "bitstream",
    "bundle2bitstream",
    "collection2item",
    "collection",
    "community2collection",
    "community2community",
    "community",
    "handle",
    "item2bundle",
    "item",
    "metadatafieldregistry",
    "metadataschemaregistry",
    "metadatavalue",
]

dataframes = [catalog.load(f"raw/dspacedb5/{table}#parquet") for table in tables]

[(table, df.shape) for table, df in zip(tables, dataframes)]


## Funcion completa

Esta celda replica la implementacion del nodo `load_dspacedb5_tables`. La idea es editar aca primero y despues copiar a `src/kedro_cic/pipelines/load_dspacedb5/nodes.py`.

In [ ]:
_EXTRACT_META_COLS = [
    "_source_system",
    "_source_table",
    "_extract_datetime",
    "_extract_date",
    "_source_label",
    "_institution_ror",
    "_extract_env",
    "_filter_param",
    "_filter_value",
]


def _add_extract_metadata(df: pd.DataFrame) -> pd.DataFrame:
    enriched_df = df.copy()
    for col in _EXTRACT_META_COLS:
        if col not in enriched_df.columns:
            enriched_df[col] = pd.NA
    enriched_df["_extract_datetime"] = pd.to_datetime(
        enriched_df["_extract_datetime"], errors="coerce"
    )
    if "_extract_date" in enriched_df.columns:
        enriched_df["_extract_date"] = pd.to_datetime(
            enriched_df["_extract_date"], errors="coerce"
        ).dt.date
    return enriched_df


def _add_load_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    enriched_df = df.copy()
    if load_datetime is None:
        load_datetime = pd.Timestamp.now(tz="UTC").floor("s").tz_localize(None)
    enriched_df["_load_datetime"] = pd.to_datetime(load_datetime)
    return enriched_df


def _astype_str(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for column in columns:
        if column in df.columns:
            df[column] = df[column].astype(str)
    return df


def _prepare_table(
    df: pd.DataFrame,
    string_columns: list[str] | None = None,
    drop_columns: list[str] | None = None,
) -> pd.DataFrame:
    prepared_df = _add_extract_metadata(df).convert_dtypes()
    if string_columns:
        prepared_df = _astype_str(prepared_df, string_columns)
    if drop_columns:
        prepared_df = prepared_df.drop(columns=drop_columns, errors="ignore")
    return _add_load_metadata(prepared_df)


def load_dspacedb5_tables(
    df_bitstream,
    df_bundle2bitstream,
    df_collection2item,
    df_collection,
    df_community2collection,
    df_community2community,
    df_community,
    df_handle,
    df_item2bundle,
    df_item,
    df_metadatafieldregistry,
    df_metadataschemaregistry,
    df_metadatavalue,
):
    return (
        _prepare_table(df_bitstream, string_columns=["uuid"]),
        _prepare_table(
            df_bundle2bitstream, string_columns=["bundle_id", "bitstream_id"]
        ),
        _prepare_table(df_collection2item, string_columns=["collection_id", "item_id"]),
        _prepare_table(
            df_collection,
            string_columns=["uuid", "submitter", "admin"],
            drop_columns=["template_item_id", "logo_bitstream_id"],
        ),
        _prepare_table(
            df_community2collection,
            string_columns=["collection_id", "community_id"],
        ),
        _prepare_table(
            df_community2community,
            string_columns=["parent_comm_id", "child_comm_id"],
        ),
        _prepare_table(
            df_community,
            string_columns=["uuid", "admin"],
            drop_columns=["logo_bitstream_id"],
        ),
        _prepare_table(df_handle, string_columns=["resource_id"]),
        _prepare_table(df_item2bundle, string_columns=["bundle_id", "item_id"]),
        _prepare_table(
            df_item,
            string_columns=["uuid", "item_id", "submitter_id", "owning_collection"],
        ),
        _prepare_table(df_metadatafieldregistry),
        _prepare_table(df_metadataschemaregistry),
        _prepare_table(df_metadatavalue, string_columns=["dspace_object_id"]),
    )


In [ ]:
loaded_tables = load_dspacedb5_tables(*dataframes)

summary = pd.DataFrame(
    {
        "table": tables,
        "rows": [len(df) for df in loaded_tables],
        "columns": [len(df.columns) for df in loaded_tables],
        "source_system": [df["_source_system"].iloc[0] if len(df) else "dspacedb5" for df in loaded_tables],
        "source_label": [df["_source_label"].iloc[0] if len(df) else pd.NA for df in loaded_tables],
        "institution_ror": [df["_institution_ror"].iloc[0] if len(df) else pd.NA for df in loaded_tables],
        "extract_datetime": [df["_extract_datetime"].iloc[0] if len(df) else pd.NA for df in loaded_tables],
        "load_datetime": [df["_load_datetime"].iloc[0] if len(df) else pd.NA for df in loaded_tables],
    }
)

summary


## Inspeccion rapida

Usar `table_to_preview` para revisar una tabla puntual antes de copiar cambios a `nodes.py`.

In [ ]:
table_to_preview = "item"
df_preview = loaded_tables[tables.index(table_to_preview)]

df_preview.head()


In [ ]:
metadata_columns = [
    "_source_system",
    "_source_table",
    "_extract_datetime",
    "_extract_date",
    "_source_label",
    "_institution_ror",
    "_extract_env",
    "_filter_param",
    "_filter_value",
    "_load_datetime",
]

df_preview[metadata_columns].head()
